# 欢迎来到第二天实验室！

## 练习目标（理念）

今天把 **Chat Completions API** 从「概念」落到「能跑的调用」：先用原始 HTTP，再用 OpenAI Python 客户端，再切换到 Gemini / Ollama 等 **OpenAI 兼容端点（OpenAI-compatible endpoints）**。

## 和本课 Day 2 的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|------------------|
| Endpoint + JSON payload | `requests.post(.../v1/chat/completions)` |
| OpenAI Python 客户端 | `OpenAI().chat.completions.create(...)` |
| 兼容端点换 `base_url` | Gemini、Ollama |
| 本地开源模型 | `llama3.2`、`deepseek-r1:1.5b` |

## 怎么跑

1. 准备 `.env`：`OPENAI_API_KEY`；可选 `GOOGLE_API_KEY`
2. 若跑 Ollama：本机先 `ollama serve`，再按需 `ollama pull`
3. 从上到下依次运行单元格


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">在我们开始之前 ——</h2>
            <span style="color:#f71;">先看一眼本课程的资源页：里面有幻灯片等有用链接。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            建议收藏；课程推进过程中这里还会继续补充链接。
            </span>
        </td>
    </tr>
</table>


## 首先 —— 聊聊 Chat Completions API

1. 调用 LLM 最简单、最常见的一种方式
2. 叫「聊天补全（Chat Completions）」，因为模型任务是：给定一段对话，预测接下来该说什么
3. 这套 API 由 OpenAI 推广开来，后来几乎成了业界通用习惯

### 我们先再次调用 OpenAI

非 OpenAI 路线的同学也别急：后面马上会用同一套客户端去打 Gemini / Ollama。


In [ ]:
# ========== 环境：加载并粗检 OPENAI_API_KEY ==========

# 导入 os：读环境变量
import os
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境
from dotenv import load_dotenv

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OpenAI 密钥（不要把真实密钥写进笔记本）
api_key = os.getenv('OPENAI_API_KEY')

# 分层提示：没密钥 / 前缀不像 sk-proj- / 看起来正常（提示文案保持英文）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


## 你知道什么是端点（Endpoint）吗？

如果还不熟，请先看 Guides 文件夹里的技术基础指南。

下面这个单元格会组装一个你可能感兴趣的 Chat Completions 请求……


In [ ]:
# ========== 原始 HTTP：准备 headers 与 JSON payload ==========

# 导入 requests：用 HTTP 直接打 REST 端点（不经过 OpenAI SDK）
import requests

# Authorization：Bearer + API Key；Content-Type 声明 JSON 正文
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

# payload：model id 与 messages 保持原样（影响实际请求）
payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

# 在笔记本里直接查看即将发送的字典
payload


In [ ]:
# ========== 原始 HTTP：POST 到 OpenAI Chat Completions ==========

# 对官方端点发 POST；json=payload 会序列化并带上正确 Content-Type
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

# 把响应体解析成 Python dict，方便下一步取值
response.json()


In [ ]:
# ========== 从 JSON 响应里取出助手回复文本 ==========

# choices[0].message.content：第一条候选里的助手内容
response.json()["choices"][0]["message"]["content"]


# `openai` 这个包是什么？

它是 **Python 客户端库（client library）**。

本质上只是对 HTTP 端点做精确封装：让你用更干净的 Python 代码，而不必手搓 JSON。

它是开源、轻量的。有人误以为里面装着 OpenAI 模型权重——**并不是**！模型仍在远端（或你指定的兼容端点）上运行。


In [ ]:
# ========== 用 OpenAI 客户端做同样的事（更干净的 Python） ==========

# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI
# 默认会读环境变量 OPENAI_API_KEY
openai = OpenAI()

# chat.completions.create：一次聊天补全；model / messages 保持原样
response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# SDK 对象上取回复文本（不必再自己挖 JSON）
response.choices[0].message.content


## 然后这件关键的事发生了：

OpenAI 的 Chat Completions API 太流行了，其他模型厂商开始提供**相同形状**的端点，叫做 **OpenAI 兼容端点（OpenAI-compatible endpoints）**。

例如 Google 提供了：https://generativelanguage.googleapis.com/v1beta/openai/

OpenAI 也允许你：继续用同一套客户端库，只改 **端点 URL（`base_url`）** 和 **密钥**，就能打到别的厂商。

示例写法：

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

需要明确：代码里虽然写着 `OpenAI`，我们只是在用这个轻量 Python 客户端去调端点——**这里并不涉及 OpenAI 的模型本身**。

若仍困惑，请看 Guides 文件夹里的 Guide 9。接下来动手试！

## 可选：试用 Google Gemini

1. 打开 https://aistudio.google.com/
2. 在 https://aistudio.google.com/api-keys 创建密钥
3. 把密钥写入 `.env` 并保存：

`GOOGLE_API_KEY=AIz...`


In [ ]:
# ========== 可选：校验 GOOGLE_API_KEY ==========

# Gemini OpenAI 兼容端点（URL 保持原样）
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# 再次加载 .env，确保刚写入的 GOOGLE_API_KEY 生效
load_dotenv(override=True)

# 读取 Google API Key
google_api_key = os.getenv("GOOGLE_API_KEY")

# 分层提示文案保持英文；没有密钥可跳过后面 2 格
if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 用同一套 SDK 调用 Gemini ==========

# base_url 指向 Google；api_key 用 GOOGLE_API_KEY
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

# model id 与 user 消息保持原样
response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# 取出助手回复文本
response.choices[0].message.content


## Ollama 也提供 OpenAI 兼容端点

……而且跑在你的**本地电脑**上！

若下一格没有打印出 Ollama 正在运行的提示，请打开终端执行 `ollama serve`（有的环境也写成课程文档里的 `ollamaserve`）。


In [ ]:
# ========== 探活：本机 Ollama 是否在监听 ==========

# GET 本地 11434；若服务起来，通常会返回类似 b'Ollama is running' 的内容
requests.get("http://localhost:11434").content


### 从 Meta 拉取 `llama3.2`

若机器较小，可改用 `llama3.2:1b`。

不要用 `llama3.3` 或 `llama4`——它们对多数学习用电脑来说太大了。


In [ ]:
# ========== 拉取本地模型权重（需本机已安装 Ollama CLI） ==========

# 感叹号：在 Jupyter 里执行 shell 命令；模型名保持原样
!ollama pull llama3.2


In [ ]:
# ========== 创建指向本地 Ollama 的 OpenAI 客户端 ==========

# 再次导入（本格可独立理解；前面若已导入也无妨）
from openai import OpenAI

# Ollama 的 OpenAI 兼容 Base URL（/v1）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key 对本地 Ollama 通常任意非空即可；这里沿用原代码的 'ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== 用本地 llama3.2 问一个问题 ==========

# Chat Completions：model / messages 内容保持原样（影响模型回答）
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "What happened on Tiananmen square in 1989?"}])

# 取出助手回复文本
response.choices[0].message.content


In [ ]:
# ========== 再拉一个小模型：deepseek-r1:1.5b ==========

# 说明：这是从阿里云 Qwen「蒸馏（distill）」得到的 DeepSeek 推理小模型（课程原注）
# shell：拉取模型，名称保持原样
!ollama pull deepseek-r1:1.5b


In [ ]:
# ========== 用 deepseek-r1:1.5b 再问同一类问题 ==========

# user 文本保持原样（含原拼写）；model id 不变
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "at happened on Tiananmen square in 1989?"}])

# 取出助手回复，便于和上一格对比风格 / 审查策略
response.choices[0].message.content


# 家庭作业

把第 1 天的「网页摘要」项目升级为：用 **Ollama 本地开源模型**，而不是云端 OpenAI。

若你不想付费调用 API，后续很多项目都可以沿用这一技术。

**好处：**
1. 无 API 按量费用——开源模型本地跑
2. 数据不离开你的机器

**坏处：**
1. 能力通常明显弱于前沿（Frontier）云端模型

## Ollama 安装回顾

访问 [ollama.com](https://ollama.com) 安装即可。

装好后，本机应已有 Ollama 服务。访问：  
[http://localhost:11434/](http://localhost:11434/)

应看到类似 “Ollama is running” 的消息。

若没有：新开终端（Mac）或 PowerShell（Windows），运行 `ollama serve`；  
另一个终端再执行 `ollama pull llama3.2`，然后重试上述地址。

若太慢，可改用 `llama3.2:1b`：`ollama pull llama3.2:1b`，并把代码里的 `MODEL = "llama3.2"` 改成 `MODEL = "llama3.2:1b"`。


In [ ]:
# ========== 作业实现：本地 Ollama 做网页辛辣摘要 ==========

# 从本地 scraper 导入抓取函数（需同目录有 scraper.py）
from scraper import fetch_website_contents
# 笔记本里用 Markdown 展示摘要
from IPython.display import Markdown, display
# OpenAI 兼容客户端（这里指向 Ollama）
from openai import OpenAI

# 若本格导入报错，请先去同目录排错笔记本（troubleshooting）

# system prompt 保持英文：决定「毒舌幽默摘要」风格；可自行实验改最后一句语言
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# user prompt 前缀保持英文：后面会拼接网页正文
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

# 本地 Ollama 的 OpenAI 兼容地址
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建客户端；api_key 对本地通常任意非空
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

def messages_for(website):
    # 组装 system + user 两条消息；user 内容 = 前缀 + 网页正文
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def summarize(url):
    # 先抓网页正文
    website = fetch_website_contents(url)
    # 再让本地 llama3.2 生成摘要；model id 保持原样
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    # 返回助手文本
    return response.choices[0].message.content

def display_summary(url):
    # 摘要 → Markdown 展示
    summary = summarize(url)
    display(Markdown(summary))

# 试跑：课程作者站点；可改成你想摘要的 URL
display_summary("https://edwarddonner.com")
